In [1]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "arxiv", "pymupdf", "transformers", "datasets", "accelerate"])

0

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", 
    "langchain", "langchain-community", "faiss-cpu", 
    "sentence-transformers", "pymupdf", "langchain-huggingface"])

0

In [3]:
import fitz  # pymupdf
import os

def load_pdfs(folder):
    docs = []
    for f in os.listdir(folder):
        if f.endswith(".pdf"):
            pdf = fitz.open(os.path.join(folder, f))
            text = ""
            for page in pdf:
                text += page.get_text()
            docs.append({"source": f, "text": text})
            print(f"Lu : {f} — {len(text)} caractères")
    return docs

docs = load_pdfs("papers")
print(f"\nTotal : {len(docs)} documents chargés")

#etape 2 :chunking
def chunk_text(text, source, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append({"source": source, "text": chunk})
        i += chunk_size - overlap  # overlap pour ne pas perdre le contexte
    return chunks

# Chunker tous les documents
all_chunks = []
for doc in docs:
    chunks = chunk_text(doc["text"], doc["source"])
    all_chunks.extend(chunks)
    print(f"{doc['source']} → {len(chunks)} chunks")

print(f"\nTotal : {len(all_chunks)} chunks prêts pour les embeddings")

Lu : main_notes.pdf — 353573 caractères
Lu : paper_0.pdf — 54321 caractères
Lu : 2005.11401v4.pdf — 69078 caractères
Lu : 1706.03762v7.pdf — 39498 caractères
Lu : cs224n-self-attention-transformers-2023_draft.pdf — 42199 caractères

Total : 5 documents chargés
main_notes.pdf → 187 chunks
paper_0.pdf → 23 chunks
2005.11401v4.pdf → 29 chunks
1706.03762v7.pdf → 18 chunks
cs224n-self-attention-transformers-2023_draft.pdf → 20 chunks

Total : 277 chunks prêts pour les embeddings


In [4]:
#l'étape 3 — transformer ces chunks en vecteurs et les indexer dans FAISS 
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

# ── Charger le modèle d'embeddings ─────────────────────────────────
print("Chargement du modèle d'embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
# Ce modèle encode chaque chunk en un vecteur de 384 dimensions

# ── Encoder tous les chunks ─────────────────────────────────────────
print(f"Encodage de {len(all_chunks)} chunks...")
texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, batch_size=32, show_progress_bar=True)

print(f"Shape des embeddings : {embeddings.shape}")
# → (277, 384) : 277 chunks, chacun = vecteur de 384 dimensions

# ── Créer l'index FAISS ─────────────────────────────────────────────
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"Index FAISS créé — {index.ntotal} vecteurs indexés")

# ── Sauvegarder sur disque ──────────────────────────────────────────
faiss.write_index(index, "corpus.index")
with open("chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print("Sauvegardé : corpus.index + chunks.pkl ✓")

Chargement du modèle d'embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encodage de 277 chunks...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Shape des embeddings : (277, 384)
Index FAISS créé — 277 vecteurs indexés
Sauvegardé : corpus.index + chunks.pkl ✓


In [5]:
from transformers import pipeline
import numpy as np

# ── Charger l'index et les chunks ──────────────────────────────────
index = faiss.read_index("corpus.index")
with open("chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

# ── Fonction de retrieval ───────────────────────────────────────────
def retrieve(question, k=3):
    q_vec = embedder.encode([question]).astype(np.float32)
    distances, indices = index.search(q_vec, k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "text": all_chunks[idx]["text"],
            "source": all_chunks[idx]["source"],
            "score": float(distances[0][i])
        })
    return results

# ── Charger le LLM pour la génération ──────────────────────────────
print("Chargement du LLM...")
generator = pipeline("text-generation", 
                     model="gpt2",
                     max_new_tokens=150,
                     pad_token_id=50256)

# ── Chaîne RAG complète ─────────────────────────────────────────────
def rag(question):
    # 1. Retrieval
    chunks = retrieve(question, k=3)
    context = "\n\n".join([c["text"] for c in chunks])
    sources = list(set([c["source"] for c in chunks]))
    
    # 2. Prompt
    prompt = f"""Answer the question based on the context below.
    
Context: {context[:1500]}

Question: {question}

Answer:"""
    
    # 3. Génération
    response = generator(prompt)[0]["generated_text"]
    
    return {
        "question": question,
        "answer": response,
        "sources": sources
    }

# ── Test sur 3 questions ────────────────────────────────────────────
questions = [
    "What is the attention mechanism in transformers?",
    "How does RAG work?",
    "What is gradient descent?"
]

print("Test du pipeline RAG\n" + "="*50)
for q in questions:
    result = rag(q)
    print(f"\nQuestion : {result['question']}")
    print(f"Réponse  : {result['answer']}")
    print(f"Sources  : {result['sources']}")
    print("-"*50)

Chargement du LLM...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test du pipeline RAG


Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : What is the attention mechanism in transformers?
Réponse  : Answer the question based on the context below.

Context: attend to “Zuko”, while “made” can attend to “Zuko” and “made”.) Intuitively, these are the biggest components to understand. How- ever, as of 2023, by far the most-used architecture in NLP is called the Transformer, introduced by [Vaswani et al., 2017], and it contains a number of components that end up being quite important. So now we’ll get into the details of that architecture. 3 The Transformer The Transformer is an architecture based on self-attention that con- sists of stacked Blocks, each of which contains self-attention and feed- forward layers, and a few other components we’ll discuss. If you’d like to take a peek for intuition, we have a diagram of a Transformer language model architecture in Figure 4. The components we haven’t gone over are multi-head self-attention, layer normalization, resid- ual connections, and attention scaling—and of course

Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : How does RAG work?
Réponse  : Answer the question based on the context below.

Context: and the RAG model would perform equivalently to BART. The collapse could be due to a less-explicit requirement for factual knowledge in some tasks, or the longer target sequences, which could result in less informative gradients for the retriever. Perez et al. [46] also found spurious retrieval results when optimizing a retrieval component in order to improve performance on downstream tasks. I Number of instances per dataset The number of training, development and test datapoints in each of our datasets is shown in Table 7. 19

2018 world leaders. Accuracy with mismatched indices is low (12% with the 2018 index and 2016 leaders, 4% with the 2016 index and 2018 leaders). This shows we can update RAG’s world knowledge by simply replacing its non-parametric memory. Effect of Retrieving more documents Models are trained with either 5 or 10 retrieved latent documents, and we do not observe si

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

questions_suggerees = [
    "What is the attention mechanism in transformers?",
    "How does RAG work?",
    "What is gradient descent?",
    "What is a neural network?",
    "What is the difference between BERT and GPT?",
    "What is backpropagation?",
    "What is a loss function?",
    "How does the transformer architecture work?",
    "What is fine-tuning?",
    "What is an embedding?"
]

# ── UI ──────────────────────────────────────────────────────────────
dropdown = widgets.Dropdown(
    options=["-- Choisir une question --"] + questions_suggerees,
    description="Suggestion :",
    layout=widgets.Layout(width="600px")
)

text_input = widgets.Text(
    placeholder="Ou tape ta propre question ici...",
    layout=widgets.Layout(width="600px")
)

button = widgets.Button(
    description="Poser la question",
    button_style="primary"
)

output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        question = text_input.value.strip()
        if not question and dropdown.value != "-- Choisir une question --":
            question = dropdown.value
        if not question:
            print("Tape une question !")
            return
        print(f"Question : {question}\n")
        print("Recherche en cours...")
        result = rag(question)
        print(f"Réponse  : {result['answer']}\n")
        print(f"Sources  : {result['sources']}")

button.on_click(on_click)

display(widgets.VBox([dropdown, text_input, button, output]))